# Scraping des avis Google Play — Mobile Money & Banques Niger

Pipeline de collecte des avis pour les 20 apps recensées.

**Points clés :**
- Pagination plafonnée (`MAX_PER_APP`) — on ne télécharge jamais plus que nécessaire, même pour les apps multi-pays (Wave, Airtel ≈ 100k+ avis)
- Checkpoints par app dans `checkpoints/` → si la session Colab crashe, relancer la cellule de scraping reprend là où on s'était arrêté
- Retry avec backoff sur chaque lot
- `reviewId` comme clé primaire + déduplication à la consolidation

In [ ]:
# 1. Installation
!pip install google-play-scraper -q

In [ ]:
# 2. Imports
from google_play_scraper import reviews, Sort
import pandas as pd
from urllib.parse import urlparse, parse_qs
from pathlib import Path
import time

In [ ]:
# 3. Configuration
MAX_PER_APP = 5000        # plafond d'avis par app
BATCH_SIZE = 200          # max supporté par l'API par requête
LANG = "fr"               # français uniquement — choix du projet
COUNTRY = "ne"            # storefront Niger
SLEEP_BETWEEN_BATCHES = 0.5
SLEEP_BETWEEN_APPS = 3
MAX_RETRIES = 3
CHECKPOINT_DIR = Path("checkpoints")
OUTPUT_FILE = "all_playstore_reviews.csv"

# app_id -> nom lisible (pour l'analyse et le dashboard)
APPS = {
    "com.airtel.africa.selfcare":        "My Airtel",
    "www.mymoov.ne":                     "MyMoovNiger",
    "com.wave.personal":                 "Wave",
    "com.ne.ifutur.imoney_public":       "iMoney Finance",
    "com.iisoft.tm.myNita":              "My Nita",
    "com.iisoft.tm.appamana":            "Amanata",
    "com.alizza.businessalizzapay_app":  "Al Izza Business Pay",
    "plus.H5F57DD7C":                    "AFUWA MONEY",
    "com.iisoft.zeynacash":              "ZeynaCash",
    "com.app.ecobank":                   "Ecobank Mobile",
    "africa.of.boamobile":               "MyBOA",
    "net.bianiger.mbanking":             "BIA Niger Mobile",
    "com.cr2.orabank":                   "My KEAZ (Orabank)",
    "com.m2i.corismoney":                "Coris Money V2",
    "com.bfi.bsic.togo":                 "BSIC NET",
    "net.banqueatlantique.mbanking":     "Atlantique Mobile",
    "com.mediasoft.bagri":               "BAGRINET",
    "com.taholding.bin":                 "BIN MOBILE",
    "com.bhnmobile.digital.prod":        "BHN Mobile",
    "com.iisoft.ccp_mobile_ng":          "CCP Mobile",
}

FIELDS = ["reviewId", "score", "content", "at", "replyContent",
          "appVersion", "userName", "thumbsUpCount"]

In [ ]:
# 4. Fonctions de scraping
def get_app_id_from_url(url: str):
    """Extrait l'appId d'une URL Play Store (utile si tu repars d'URLs)."""
    return parse_qs(urlparse(url).query).get("id", [None])[0]


def fetch_reviews_capped(app_id: str, max_count: int) -> list:
    """Récupère au plus max_count avis via pagination, avec retry par lot."""
    collected, token = [], None
    while len(collected) < max_count:
        remaining = max_count - len(collected)
        for attempt in range(1, MAX_RETRIES + 1):
            try:
                batch, token = reviews(
                    app_id,
                    lang=LANG,
                    country=COUNTRY,
                    sort=Sort.NEWEST,
                    count=min(BATCH_SIZE, remaining),
                    continuation_token=token,
                )
                break
            except Exception as e:
                if attempt == MAX_RETRIES:
                    raise
                wait = 5 * attempt
                print(f"      ⚠️ lot en échec ({e}), retry dans {wait}s "
                      f"[{attempt}/{MAX_RETRIES}]")
                time.sleep(wait)

        if not batch:                      # plus rien à récupérer
            break
        collected.extend(batch)
        print(f"      … {len(collected)} avis", end="\r")
        if token is None or token.token is None:  # fin de la pagination
            break
        time.sleep(SLEEP_BETWEEN_BATCHES)
    return collected[:max_count]


def scrape_app(app_id: str, app_name: str) -> pd.DataFrame:
    raw = fetch_reviews_capped(app_id, MAX_PER_APP)
    rows = []
    for r in raw:
        row = {k: r.get(k) for k in FIELDS}
        row["at"] = row["at"].isoformat() if row["at"] else None
        row["app_id"] = app_id
        row["app_name"] = app_name
        rows.append(row)
    return pd.DataFrame(rows)

## Lancement du scraping

Relançable sans risque : les apps déjà présentes dans `checkpoints/` sont skippées.

In [ ]:
# 5. Scraping (relançable après un crash Colab)
CHECKPOINT_DIR.mkdir(exist_ok=True)
print(f"🚀 Scraping de {len(APPS)} apps (max {MAX_PER_APP} avis/app)\n")

for i, (app_id, app_name) in enumerate(APPS.items(), 1):
    ckpt = CHECKPOINT_DIR / f"{app_id}.csv"
    if ckpt.exists():
        print(f"[{i}/{len(APPS)}] {app_name} — déjà fait, skip ✔")
        continue

    print(f"[{i}/{len(APPS)}] {app_name} ({app_id}) ...")
    try:
        df_app = scrape_app(app_id, app_name)
        df_app.to_csv(ckpt, index=False, encoding="utf-8-sig")
        print(f"   ✅ {len(df_app)} avis sauvegardés → {ckpt}")
    except Exception as e:
        print(f"   ❌ Erreur définitive : {e}")
    time.sleep(SLEEP_BETWEEN_APPS)

In [ ]:
# 6. Consolidation finale + déduplication
parts = [pd.read_csv(f) for f in sorted(CHECKPOINT_DIR.glob("*.csv"))]
assert parts, "Aucune donnée collectée — lance d'abord la cellule de scraping."

df = pd.concat(parts, ignore_index=True)
before = len(df)
df = df.drop_duplicates(subset="reviewId")
df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")

print(f"🎉 Total : {len(df)} avis ({before - len(df)} doublons retirés)")
print(f"Fichier : {OUTPUT_FILE}\n")
print("Résumé par app :")
print(df.groupby("app_name").size().sort_values(ascending=False).to_string())

In [ ]:
# 7. Aperçu rapide
display(df.head())
df["score"].value_counts().sort_index()

## Notes

- **Périmètre linguistique : français uniquement** (`lang='fr'`). C'est un choix assumé du projet — toute l'analyse aval (tagging thématique, rapport, dashboard) est conçue pour des avis en français. Ne pas relancer en `en`.
- L'API n'expose pas le pays du reviewer : le mélange de pays sur les apps multi-pays (Wave, Airtel, Ecobank…) est documenté comme limite dans le rapport.
